# Explore the Yelp data (McDonald's and IKEA)

Before a model reads any review, we look at the data: which locations are reviewed, how the ratings are distributed, and who writes reviews.

In this notebook, we will:
1. Load the McDonald's and IKEA locations and compare their ratings.
2. Load all IKEA reviews and explore stars, time, stores and review length.
3. Read some reviews and search for a topic.
4. Prepare a small, anonymized sample of 500 reviews for notebooks 03–05.

**Time:** about 15 minutes. This notebook needs no API key and costs nothing.

**Before you start: get the data from Moodle**
1. Download `llm_fundamentals_data.zip` from Moodle (Week 10).
2. Drag the zip file into the file Explorer, onto the folder `data`.
3. Run the cells below: the cell **Unzip the data** unpacks the zip into `data`.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import zipfile
from datetime import datetime
from platform import python_version

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Course helpers: DATA is the folder 'data', anonymize() removes personal details
from llm_course import DATA, anonymize

# Show current working directory
print(os.getcwd())

## Unzip the data

Unpacks `data/llm_fundamentals_data.zip` from Moodle into `data`. Only needed once.

In [ ]:
# Zip file from Moodle, dragged onto the folder data
ZIP_FILE = DATA / 'llm_fundamentals_data.zip'

if os.path.exists(DATA / 'ikea_reviews.csv'):
    print('Data already unzipped.')
elif os.path.exists(ZIP_FILE):
    with zipfile.ZipFile(ZIP_FILE) as z:
        z.extractall(DATA)
        print('Unzipped into data:', z.namelist())
else:
    print('Zip file not found: download llm_fundamentals_data.zip from Moodle (Week 10),\n'
          'drag it onto the folder data, then run this cell again.')

## Part 1: The locations

### Import the business records

In [ ]:
# Read the McDonald's and IKEA locations
df_bus = pd.read_csv(DATA / 'businesses_mcdonalds_ikea.csv')

# Shape and first rows
print(df_bus.shape)
df_bus.head(5)

### Number of locations and reviews per chain

In [ ]:
# Locations, reviews and mean rating per chain
summary = df_bus.groupby('chain').agg(
    locations=('business_id', 'count'),
    reviews=('review_count', 'sum'),
    mean_stars=('stars', 'mean'),
)
summary.round(2)

### Average rating: per location vs. per review

`mean_stars` counts every location once. A location with 500 reviews should count more than one with 5 reviews, so we weight each location by its number of reviews.

In [ ]:
# Weighted average: sum of (stars x reviews) divided by the sum of reviews
df_bus['stars_x_reviews'] = df_bus['stars'] * df_bus['review_count']
totals = df_bus.groupby('chain')[['stars_x_reviews', 'review_count']].sum()
summary['weighted_stars'] = totals['stars_x_reviews'] / totals['review_count']

summary.round(2)

### Distribution of the location ratings

In [ ]:
# Share of locations per star rating, for each chain
shares = pd.crosstab(df_bus['stars'], df_bus['chain'], normalize='columns')

shares.plot.bar(figsize=(8, 4))
plt.title('Location ratings by chain')
plt.xlabel('Stars of the location (rounded to 0.5)')
plt.ylabel('Share of locations')
plt.xticks(rotation=0)
plt.show()

### The IKEA locations

In [ ]:
# All IKEA records: 6 stores and 2 store restaurants
df_bus.loc[df_bus['chain'] == 'IKEA', ['name', 'city', 'state', 'stars', 'review_count']]

## Part 2: The IKEA reviews

### Import the IKEA reviews

In [ ]:
# Read all IKEA reviews
df = pd.read_csv(DATA / 'ikea_reviews.csv', parse_dates=['date'])

# Shape and first rows
print(df.shape)
df.head(5)

### Star distribution of the reviews

In [ ]:
# Number of reviews per star rating
counts = df['stars'].value_counts().sort_index()

ax = counts.plot.bar(color='steelblue', figsize=(6, 4))
ax.bar_label(ax.containers[0])
plt.title(f'All {len(df):,} IKEA reviews by rating')
plt.xlabel('Stars')
plt.ylabel('Number of reviews')
plt.xticks(rotation=0)
plt.show()

Many 4s and 5s, many 1s, few in between: the typical **J-shape** of online reviews. People write reviews when they are very happy or very unhappy. Even a perfect summary of these reviews describes the people who write reviews, not all customers.

### Reviews over time

In [ ]:
# Number of reviews per year
df['year'] = df['date'].dt.year
per_year = df['year'].value_counts().sort_index()

per_year.plot.bar(color='steelblue', figsize=(8, 4))
plt.title('IKEA reviews per year')
plt.xlabel('Year')
plt.ylabel('Number of reviews')
plt.show()

### Ratings by store

In [ ]:
# Number of reviews and mean rating per store
by_store = df.groupby(['store', 'city']).agg(
    reviews=('review_id', 'count'),
    mean_stars=('stars', 'mean'),
)
by_store.sort_values('reviews', ascending=False).round(2)

### Review length

In [ ]:
# Number of words per review
df['words'] = df['text'].str.split().str.len()
print(df['words'].describe().round(1))

# Review length by star rating
df.boxplot(column='words', by='stars', showfliers=False, figsize=(6, 4))
plt.suptitle('')
plt.title('Review length by stars (without outliers)')
plt.xlabel('Stars')
plt.ylabel('Words')
plt.show()

## Part 3: Read reviews

Numbers hide what people actually write. Read at least one review per star rating.

In [ ]:
# One random review for each star rating
for stars in [1, 2, 3, 4, 5]:
    review = df[df['stars'] == stars].sample(1, random_state=42).iloc[0]
    print(stars, 'stars |', review['store'], '|', review['city'])
    print(review['text'][:500], '\n')

### Search for a topic

In [ ]:
# TODO: try other topics, e.g. 'assembly', 'return', 'parking', 'staff'
topic = 'meatball'

hits = df[df['text'].str.contains(topic, case=False)]
print(f"{len(hits)} reviews mention '{topic}'")
print(f"Mean stars: {hits['stars'].mean():.2f} (all reviews: {df['stars'].mean():.2f})")
hits[['stars', 'store', 'text']].head(5)

## Part 4: Prepare a sample for the API notebooks

Reviews can contain names, phone numbers and other personal details. **We never send personal data to an external API.** Notebooks 03–05 therefore use a prepared sample:

1. Draw a random sample of **500 reviews**. That is enough for all exercises and keeps the notebooks fast.
2. Remove personal details with `anonymize()`: names of people, e-mail addresses and phone numbers. It runs on this computer (libraries Presidio and spaCy), not through an API. Company names, places and dates stay, because the model needs them to understand a review.
3. Check the result and save it.

### How `anonymize()` works: one example

In [ ]:
# A made-up text with a name, a phone number and an e-mail address
example = ('I asked for John Smith at the IKEA store in Tampa. '
           'Call me on 813-555-0199 or write to john.smith@example.com.')

print(example)
print(anonymize(example))

### Draw the sample

Some review texts appear more than once in the data. We keep each text once. We also leave out the three example reviews that notebook 03 shows to the model: a review used as an example must not be used again to test the model.

In [ ]:
# Reviews whose text appears more than once
duplicates = df[df.duplicated('text', keep=False)]
print('Rows with a text that appears more than once:', len(duplicates))

# Keep each text once and leave out the three example reviews of notebook 03
examples = pd.read_json(DATA / 'fewshot_examples.json')
unique = df.drop_duplicates('text')
unique = unique[~unique['text'].isin(examples['text'])]
print('All reviews:', len(df), '| unique reviews without the examples:', len(unique))

In [ ]:
# Random sample of 500 reviews. The fixed random_state gives everybody the same sample.
sample = unique.sample(500, random_state=42)
sample = sample[['store', 'city', 'stars', 'date', 'text']].reset_index(drop=True)

# Short IDs R001 ... R500: easy to read and to cite later
sample.insert(0, 'id', [f'R{i:03d}' for i in range(1, 501)])

sample['stars'].value_counts().sort_index()

### Anonymize the sample

In [ ]:
# Keep the original text for the comparison below, then anonymize (about 20 seconds)
sample['original'] = sample['text']
sample['text'] = [anonymize(text) for text in tqdm(sample['original'])]

changed = sample[sample['text'] != sample['original']]
print('Reviews with at least one replacement:', len(changed), 'of', len(sample))

### Check the result

Automatic anonymization makes mistakes: it can miss a name, and it can remove harmless words. Compare some texts before and after.

In [ ]:
# Before and after, for five changed reviews
pd.set_option('display.max_colwidth', 400)
changed[['id', 'original', 'text']].head(5)

**Your turn:** read a few more rows, for example with `changed[['id', 'original', 'text']].sample(5)`. If you still find a personal detail, replace the text of this review in the next cell.

In [ ]:
# TODO (only if needed): correct a text by hand, e.g.
# sample.loc[sample['id'] == 'R007', 'text'] = 'your corrected text'

### Save the sample

We save only the anonymized texts. The file stays in `data/processed/`, which Git ignores.

In [ ]:
# Save the sample without the original texts
os.makedirs(DATA / 'processed', exist_ok=True)
sample.drop(columns='original').to_csv(DATA / 'processed/reviews_sample.csv', index=False)

print('Saved', len(sample), 'reviews: data/processed/reviews_sample.csv')

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')